## 1. Purpose, privacy, and controlled comparison

# Compare Qwen3, BGE-M3, and optional Gemini embeddings

The notebook uses the same six deduplicated recordings, timestamped passages, and BERTopic settings for every model. Set explicit canonical episode IDs below to override deterministic selection.

**Privacy:** Qwen and BGE run locally. If you enable Gemini, selected passage text is transmitted to Google's API. Outputs contain transcript excerpts and audio links. Automatic metrics describe cluster structure, not topic quality; listen to sources and complete manual review before drawing conclusions.

## 2. Imports and display configuration

In [ ]:
from dataclasses import asdict
from datetime import UTC, datetime
from itertools import combinations
import os
from pathlib import Path

from IPython.display import display
import pandas as pd

from arvamusfestivali_transcripts.topic_analysis import (
    ChunkingConfig, DeterministicHashEmbedder, TopicModelConfig,
    available_embedders, build_review_rows, chunk_episodes, compare_runs,
    embed_passages, evaluate_run, export_experiment, fit_topic_model,
    load_corpus, plot_episode_timeline, plot_episode_topic_heatmap,
    plot_semantic_map, plot_topic_correspondence, plot_topic_sizes,
    select_diverse_episodes,
)
from arvamusfestivali_transcripts.topic_analysis.cache import CacheIdentity, EmbeddingCache

pd.set_option("display.max_colwidth", 160)

## 3. User configuration

In [ ]:
PROJECT_ROOT = Path(os.environ.get("TOPIC_ANALYSIS_PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "data" / "transcripts").exists() and (PROJECT_ROOT.parent / "data" / "transcripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

YEAR = 2026
EPISODE_COUNT = 6
EXPLICIT_EPISODE_IDS: tuple[str, ...] = ()
MODEL_KEYS = ("qwen", "bge", "gemini")
DEVICE = "cpu"
BATCH_SIZES = {"qwen": 4, "bge": 4, "gemini": 1}
CACHE_ROOT = PROJECT_ROOT / "data" / "topic-analysis" / "cache"
RESULT_ROOT = PROJECT_ROOT / "data" / "topic-analysis" / "results"
CHUNKING = ChunkingConfig()
TOPIC_MODEL = TopicModelConfig(random_state=42)
DRY_RUN_WITH_FAKE_EMBEDDINGS = False

## 4. Corpus loading, deduplication, and six-episode selection

In [ ]:
episodes = load_corpus(PROJECT_ROOT / "data" / "transcripts", YEAR)
if not episodes:
    raise ValueError(f"No validated transcripts found for {YEAR}")
selected = select_diverse_episodes(
    episodes, count=EPISODE_COUNT, explicit_ids=EXPLICIT_EPISODE_IDS or None
)
display(pd.DataFrame([
    {"episode_id": e.episode_id, "title": e.title, "audio_sha256": e.audio_sha256,
     "duplicate_episode_ids": ", ".join(e.duplicate_episode_ids),
     "source_paths": ", ".join(map(str, e.source_paths))}
    for e in selected
]))
assert len({e.audio_sha256 for e in selected}) == len(selected)

## 5. Passage construction and diagnostics

In [ ]:
passages = chunk_episodes(selected, CHUNKING)
if not passages:
    raise ValueError("Selection produced no passages; inspect transcript cues and chunking.")
display(pd.DataFrame([
    {"episode_id": e.episode_id,
     "passage_count": sum(p.episode_id == e.episode_id for p in passages),
     "word_count": sum(p.word_count for p in passages if p.episode_id == e.episode_id)}
    for e in selected
]))
print(f"{len(passages)} passages from {len(selected)} recordings")

## 6. Adapter availability, device, and cache configuration

In [ ]:
api_key = os.environ.get("GEMINI_API_KEY", "").strip()
if "gemini" in MODEL_KEYS and not api_key:
    print("Gemini skipped: GEMINI_API_KEY is not set")
if DRY_RUN_WITH_FAKE_EMBEDDINGS:
    adapters = {}
    for model_key in MODEL_KEYS:
        if model_key == "gemini" and not api_key:
            continue
        fake = DeterministicHashEmbedder(dimension=32)
        fake.key = model_key
        fake.model_id = f"deterministic-hash-{model_key}"
        adapters[model_key] = fake
    active_topic_model = TopicModelConfig(
        n_neighbors=3, n_components=2, min_cluster_size=2, min_samples=1, random_state=42
    )
else:
    adapters = dict(available_embedders())
    for model_key in ("qwen", "bge"):
        if model_key in adapters:
            adapters[model_key].device = DEVICE
    active_topic_model = TOPIC_MODEL
cache = EmbeddingCache(CACHE_ROOT)
print(f"Device: {DEVICE}; cache: {CACHE_ROOT}; results: {RESULT_ROOT}")

## 7. Embedding generation with per-model isolation

In [ ]:
embeddings = {}
cache_identities = {}
for model_key in MODEL_KEYS:
    if model_key not in adapters:
        continue
    adapter = adapters[model_key]
    try:
        result = embed_passages(
            adapter, passages, cache, BATCH_SIZES[model_key], chunking=CHUNKING
        )
    except Exception as error:
        print(f"{model_key} embedding failed: {error}")
        print("Remediation: check model access, connectivity, memory, device and batch size; "
              "for Gemini also check key and quota. Cached passages remain reusable.")
        continue
    embeddings[model_key] = result
    cache_identities[model_key] = CacheIdentity(
        model_id=adapter.model_id, model_revision=adapter.model_revision,
        dimension=adapter.dimension, instruction=adapter.instruction,
        chunking=CHUNKING, adapter_version=1,
    )
    print(f"{model_key}: {result.embeddings.shape}, {result.cache_hits} cache hits")
if not embeddings:
    raise RuntimeError("No embeddings succeeded; resolve the reported failures and rerun.")

## 8. Controlled BERTopic fitting

In [ ]:
runs = {}
for model_key, result in embeddings.items():
    try:
        runs[model_key] = fit_topic_model(passages, result, active_topic_model)
    except Exception as error:
        print(f"{model_key} BERTopic fit failed: {error}")
        print("Remediation: inspect passage count, memory, and the shared TOPIC_MODEL settings.")
if not runs:
    raise RuntimeError("No BERTopic fits succeeded; resolve the reported failures and rerun.")
embeddings = {key: embeddings[key] for key in runs}
cache_identities = {key: cache_identities[key] for key in runs}
print(f"Shared BERTopic configuration: {asdict(active_topic_model)}")

## 9. Automatic metric table

In [ ]:
metrics = pd.DataFrame([evaluate_run(runs[key], embeddings[key]) for key in runs])
display(metrics)

## 10. Semantic maps and topic-size charts

In [ ]:
for run in runs.values():
    display(plot_semantic_map(run, passages))
    display(plot_topic_sizes(run, passages))

## 11. Representative and disputed passages

In [ ]:
by_id = {p.passage_id: p for p in passages}
representatives = pd.DataFrame([
    {"model_key": key, "topic_id": topic_id, "passage_id": passage_id,
     "episode_id": by_id[passage_id].episode_id, "text": by_id[passage_id].text,
     "audio_link": by_id[passage_id].timestamped_audio_url}
    for key, run in runs.items()
    for topic_id, ids in run.representative_passages.items()
    for passage_id in ids
])
display(representatives)
# Topic numbers have no cross-model meaning. Different outlier decisions flag review cases.
disputed = pd.DataFrame([
    {"passage_id": p.passage_id, "episode_id": p.episode_id,
     "outlier_by_model": {key: int(run.topics[i]) == -1 for key, run in runs.items()},
     "text": p.text, "audio_link": p.timestamped_audio_url}
    for i, p in enumerate(passages)
    if len({int(run.topics[i]) == -1 for run in runs.values()}) > 1
])
display(disputed)

## 12. Episode heatmaps and timelines

In [ ]:
for run in runs.values():
    display(plot_episode_topic_heatmap(run, passages, normalized=True))
    display(plot_episode_timeline(run, passages))

## 13. Cross-model correspondence

In [ ]:
cross_model = compare_runs(runs, embeddings)
display(cross_model)
for left, right in combinations(runs, 2):
    display(plot_topic_correspondence({left: runs[left], right: runs[right]}, passages))

## 14. Manual review DataFrame

In [ ]:
manual_review = build_review_rows(runs, passages)
display(manual_review)
print("Review timestamped audio. Fill verdict with coherent, mixed, duplicate, or unclear "
      "and add notes in the exported manual-review.csv.")

## 15. Reproducible export

In [ ]:
experiment_id = f"topic-compare-{YEAR}-{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
export_dir = export_experiment(
    RESULT_ROOT, experiment_id, passages, runs, embeddings,
    chunking=CHUNKING, topic_model=active_topic_model,
    cache_identities=cache_identities,
)
print(f"Experiment exported to {export_dir}")
print(f"Manifest: {export_dir / 'manifest.json'}")